# rl-tunix SWE-bench Warm Pools — Demo

This notebook walks through using **Agent Sandbox warm pools** to run
[SWE-bench](https://www.swebench.com/) tasks the way the
[rl-tunix](https://github.com/google/tunix) RL/eval pipeline does: pre-warm pools
of sandboxes per task image, claim one sandbox per task, run a command inside it,
and tear down.

It exercises the three warm-pool strategies (`none`, `naive`, `sliding`) from
this example's `rl_simple_scripts/strategies.py` against a live cluster, using
**real** `R2E-Gym/SWE-Bench-Verified` images.

**Prerequisites** (see `README.md`): a cluster with the Agent Sandbox controller
+ extensions installed, `pip install -r rl_simple_scripts/requirements.txt`, and a
kubeconfig pointed at the cluster. SWE-bench images are multi-GB, so keep the task
count small and allow time for the first pull.

In [ ]:
# Install deps (uncomment on first run)
# %pip install -r requirements.txt

## 1. Configuration
Kept small for a demo. Set `NODE_SELECTOR_*` to pin to a pool, `RUNTIME_CLASS=gvisor` for isolation.

In [2]:
import logging, os
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")

NAMESPACE = "rl-tunix-swebench"
TASKS_LIMIT = 2                 # tasks to pull from the dataset
MAX_WARMPOOL_SIZE = 1          # replicas cap per image pool (tiny for the demo)
WINDOW_SIZE = 1               # sliding-window width
SANDBOX_READY_TIMEOUT = 900   # multi-GB image pulls are slow on first use

# Optional cluster pinning / isolation:
NODE_SELECTOR = None          # e.g. {"cloud.google.com/gke-nodepool": "standard-pool"}
RUNTIME_CLASS = None          # e.g. "gvisor"
IMAGE_PULL_SECRET = None      # e.g. "dockerhub-pro"

## 2. Connect to the cluster

In [ ]:
from kubernetes import client, config
from k8s_agent_sandbox import SandboxClient

import strategies, warmpool as wp

config.load_kube_config()
custom_api = client.CustomObjectsApi()
core_api = client.CoreV1Api()
sandbox_client = SandboxClient()

# Make sure the namespace exists.
try:
    core_api.create_namespace(client.V1Namespace(metadata=client.V1ObjectMeta(name=NAMESPACE)))
    print("created namespace", NAMESPACE)
except client.ApiException as e:
    print("namespace:", "exists" if e.status == 409 else e.status)

## 3. Inspect real SWE-bench task entries
Each entry carries a real `docker_image` (the repo checked out at a commit).

In [4]:
from datasets import load_dataset

ds = load_dataset("R2E-Gym/SWE-Bench-Verified", split="test")
entries = [
    {"instance_id": r["instance_id"], "docker_image": r["docker_image"], "repo": r.get("repo", "")}
    for r in ds
][:TASKS_LIMIT]

for e in entries:
    print(e["instance_id"], "->", e["docker_image"])

/Users/glottman/dev/vscode/rl-tunix/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-06-18 19:41:08,295 INFO httpx HTTP Request: HEAD https://huggingface.co/datasets/R2E-Gym/SWE-Bench-Verified/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"


2026-06-18 19:41:08,324 INFO httpx HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/R2E-Gym/SWE-Bench-Verified/1fe83d7d3cb55a5eac714155f360614b3b7c2ad2/README.md "HTTP/1.1 200 OK"


2026-06-18 19:41:08,436 INFO httpx HTTP Request: HEAD https://huggingface.co/datasets/R2E-Gym/SWE-Bench-Verified/resolve/1fe83d7d3cb55a5eac714155f360614b3b7c2ad2/SWE-Bench-Verified.py "HTTP/1.1 404 Not Found"


2026-06-18 19:41:08,782 INFO httpx HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/R2E-Gym/SWE-Bench-Verified/R2E-Gym/SWE-Bench-Verified.py "HTTP/1.1 404 Not Found"


...

2026-06-18 19:41:09,360 INFO httpx HTTP Request: GET https://huggingface.co/api/datasets/R2E-Gym/SWE-Bench-Verified/tree/1fe83d7d3cb55a5eac714155f360614b3b7c2ad2/data?recursive=true&expand=false "HTTP/1.1 200 OK"


2026-06-18 19:41:09,456 INFO httpx HTTP Request: GET https://huggingface.co/api/datasets/R2E-Gym/SWE-Bench-Verified/tree/1fe83d7d3cb55a5eac714155f360614b3b7c2ad2?recursive=false&expand=false "HTTP/1.1 200 OK"


2026-06-18 19:41:09,586 INFO httpx HTTP Request: HEAD https://huggingface.co/datasets/R2E-Gym/SWE-Bench-Verified/resolve/1fe83d7d3cb55a5eac714155f360614b3b7c2ad2/dataset_infos.json "HTTP/1.1 404 Not Found"


2026-06-18 19:41:09,587 WARNING huggingface_hub.utils._http Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


astropy__astropy-12907 -> slimshetty/swebench-verified:sweb.eval.x86_64.astropy__astropy-12907
astropy__astropy-13033 -> slimshetty/swebench-verified:sweb.eval.x86_64.astropy__astropy-13033


## 4. Define how each task is processed
Claim a pre-warmed sandbox from the image's pool, exec a probe command (router-free, via `kubectl exec`), then terminate.

In [5]:
results = []

mgr = strategies.WarmPoolManager(
    custom_api, NAMESPACE,
    node_selector=NODE_SELECTOR,
    image_pull_secret=IMAGE_PULL_SECRET,
    runtime_class=RUNTIME_CLASS,
    ready_timeout=SANDBOX_READY_TIMEOUT,
)

PROBE = ["bash", "-lc", "echo READY $(hostname); git -C /testbed log -1 --oneline 2>/dev/null || ls /"]

def process(task, pool):
    sb = sandbox_client.create_sandbox(warmpool=pool, namespace=NAMESPACE,
                                       sandbox_ready_timeout=SANDBOX_READY_TIMEOUT)
    try:
        pod = sb.get_pod_name()
        out = wp.exec_in_pod(core_api, pod, NAMESPACE, PROBE)
        print(f"[{task['instance_id']}] pod={pod}: {out.strip()}")
        results.append({"instance_id": task["instance_id"], "pod": pod, "output": out.strip()})
    finally:
        sb.terminate()

## 5. Strategy: `naive`
Pre-warm a pool for **every** unique image up front, run all tasks, tear down.

In [6]:
results.clear()
strategies.run_naive(entries, mgr, process, max_warmpool_size=MAX_WARMPOOL_SIZE)
results

2026-06-18 19:41:09,821 INFO rl-tunix-swebench.strategies [naive] pre-warming slimshetty/swebench-verified:sweb.eval.x86_64.astropy__astropy-12907 (replicas=1)


2026-06-18 19:41:10,215 INFO rl-tunix-swebench.warmpool Creating SandboxTemplate 'r2e-img-a8d0235275f3' for image slimshetty/swebench-verified:sweb.eval.x86_64.astropy__astropy-12907


2026-06-18 19:41:10,516 INFO rl-tunix-swebench.warmpool Created SandboxWarmPool 'pool-r2e-img-a8d0235275f3' (replicas=1)


2026-06-18 19:41:10,652 INFO rl-tunix-swebench.warmpool WarmPool 'pool-r2e-img-a8d0235275f3': 0/1 ready


2026-06-18 19:41:15,807 INFO rl-tunix-swebench.warmpool WarmPool 'pool-r2e-img-a8d0235275f3': 1/1 ready


...

2026-06-18 19:41:25,697 INFO rl-tunix-swebench.warmpool Deleted SandboxWarmPool 'pool-r2e-img-a8d0235275f3'


2026-06-18 19:41:25,869 INFO rl-tunix-swebench.warmpool Deleted SandboxTemplate 'r2e-img-a8d0235275f3'


2026-06-18 19:41:26,019 INFO rl-tunix-swebench.warmpool Deleted SandboxWarmPool 'pool-r2e-img-e495aee34002'


2026-06-18 19:41:26,167 INFO rl-tunix-swebench.warmpool Deleted SandboxTemplate 'r2e-img-e495aee34002'


[{'instance_id': 'astropy__astropy-12907',
  'pod': 'pool-r2e-img-a8d0235275f3-vr4nq',
  'output': 'READY pool-r2e-img-a8d0235275f3-vr4nq\nd16bfe05a7 Merge pull request #12900 from Cadair/custom_compound_model'},
 {'instance_id': 'astropy__astropy-13033',
  'pod': 'pool-r2e-img-e495aee34002-5n4bv',
  'output': 'READY pool-r2e-img-e495aee34002-5n4bv\n298ccb478e Merge pull request #13031 from meeseeksmachine/auto-backport-of-pr-13027-on-main'}]

## 6. Strategy: `sliding`
Sort by image and keep only `WINDOW_SIZE` pools warm at a time, rolling forward
as each image's tasks finish. Best for large, image-diverse batches.

In [7]:
results.clear()
strategies.run_sliding(entries, mgr, process, window_size=WINDOW_SIZE, max_warmpool_size=MAX_WARMPOOL_SIZE)
results

2026-06-18 19:41:26,178 INFO rl-tunix-swebench.strategies [sliding] pre-warming slimshetty/swebench-verified:sweb.eval.x86_64.astropy__astropy-12907 (replicas=1)


2026-06-18 19:41:26,279 INFO rl-tunix-swebench.warmpool Creating SandboxTemplate 'r2e-img-a8d0235275f3' for image slimshetty/swebench-verified:sweb.eval.x86_64.astropy__astropy-12907


2026-06-18 19:41:26,503 INFO rl-tunix-swebench.warmpool Created SandboxWarmPool 'pool-r2e-img-a8d0235275f3' (replicas=1)


2026-06-18 19:41:26,602 INFO rl-tunix-swebench.warmpool WarmPool 'pool-r2e-img-a8d0235275f3': 0/1 ready


2026-06-18 19:41:31,712 INFO rl-tunix-swebench.warmpool WarmPool 'pool-r2e-img-a8d0235275f3': 1/1 ready


...

2026-06-18 19:41:41,158 INFO root Terminated SandboxClaim: sandbox-claim-6e70f120


2026-06-18 19:41:41,159 INFO rl-tunix-swebench.strategies [sliding] image slimshetty/swebench-verified:sweb.eval.x86_64.astropy__astropy-13033 done, sliding window


2026-06-18 19:41:41,285 INFO rl-tunix-swebench.warmpool Deleted SandboxWarmPool 'pool-r2e-img-e495aee34002'


2026-06-18 19:41:41,411 INFO rl-tunix-swebench.warmpool Deleted SandboxTemplate 'r2e-img-e495aee34002'


[{'instance_id': 'astropy__astropy-12907',
  'pod': 'pool-r2e-img-a8d0235275f3-q2ctd',
  'output': 'READY pool-r2e-img-a8d0235275f3-q2ctd\nd16bfe05a7 Merge pull request #12900 from Cadair/custom_compound_model'},
 {'instance_id': 'astropy__astropy-13033',
  'pod': 'pool-r2e-img-e495aee34002-d2lxr',
  'output': 'READY pool-r2e-img-e495aee34002-d2lxr\n298ccb478e Merge pull request #13031 from meeseeksmachine/auto-backport-of-pr-13027-on-main'}]

## 7. Strategy: `none`
No pre-warming — a size-1 pool is provisioned on demand per task. Highest
per-task latency, lowest idle cost.

In [8]:
results.clear()
strategies.run_none(entries, mgr, process)
results

2026-06-18 19:41:41,418 INFO rl-tunix-swebench.strategies [none 1/2] provisioning on demand for slimshetty/swebench-verified:sweb.eval.x86_64.astropy__astropy-12907


2026-06-18 19:41:41,516 INFO rl-tunix-swebench.warmpool Creating SandboxTemplate 'r2e-img-a8d0235275f3' for image slimshetty/swebench-verified:sweb.eval.x86_64.astropy__astropy-12907


2026-06-18 19:41:41,740 INFO rl-tunix-swebench.warmpool Created SandboxWarmPool 'pool-r2e-img-a8d0235275f3' (replicas=1)


2026-06-18 19:41:41,849 INFO rl-tunix-swebench.warmpool WarmPool 'pool-r2e-img-a8d0235275f3': 0/1 ready


2026-06-18 19:41:46,963 INFO rl-tunix-swebench.warmpool WarmPool 'pool-r2e-img-a8d0235275f3': 1/1 ready


...

2026-06-18 19:42:02,922 INFO root Terminated SandboxClaim: sandbox-claim-c4dc8615


[astropy__astropy-13033] pod=pool-r2e-img-e495aee34002-7gs2n: READY pool-r2e-img-e495aee34002-7gs2n
298ccb478e Merge pull request #13031 from meeseeksmachine/auto-backport-of-pr-13027-on-main


2026-06-18 19:42:03,045 INFO rl-tunix-swebench.warmpool Deleted SandboxWarmPool 'pool-r2e-img-e495aee34002'


2026-06-18 19:42:03,170 INFO rl-tunix-swebench.warmpool Deleted SandboxTemplate 'r2e-img-e495aee34002'


[{'instance_id': 'astropy__astropy-12907',
  'pod': 'pool-r2e-img-a8d0235275f3-zlslr',
  'output': 'READY pool-r2e-img-a8d0235275f3-zlslr\nd16bfe05a7 Merge pull request #12900 from Cadair/custom_compound_model'},
 {'instance_id': 'astropy__astropy-13033',
  'pod': 'pool-r2e-img-e495aee34002-7gs2n',
  'output': 'READY pool-r2e-img-e495aee34002-7gs2n\n298ccb478e Merge pull request #13031 from meeseeksmachine/auto-backport-of-pr-13027-on-main'}]

## 8. Cleanup
The strategies tear down their own pools; this removes anything left over.

In [9]:
# Cleanup. The strategies tear down their own pools; this removes any leftovers.
# Note: SandboxWarmPool/Claim/Template are in the *extensions* group, while the
# core Sandbox CRD is in the agents.x-k8s.io group.
TARGETS = [
    ("extensions.agents.x-k8s.io", "sandboxwarmpools"),
    ("extensions.agents.x-k8s.io", "sandboxclaims"),
    ("extensions.agents.x-k8s.io", "sandboxtemplates"),
    ("agents.x-k8s.io", "sandboxes"),
]
for group, plural in TARGETS:
    objs = custom_api.list_namespaced_custom_object(group, "v1beta1", NAMESPACE, plural).get("items", [])
    for o in objs:
        name = o["metadata"]["name"]
        try:
            custom_api.delete_namespaced_custom_object(
                group, "v1beta1", NAMESPACE, plural, name,
                body=client.V1DeleteOptions(grace_period_seconds=0))
            print("deleted", plural, name)
        except client.ApiException as e:
            if e.status != 404:
                raise
print("done")

done
